# Control Feature Visualization V3

Runnable audit harness for the v3 control feature contract.

In [1]:
from __future__ import annotations

from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import display


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "requirements.txt").exists() and (candidate / "train").exists():
            return candidate
    raise FileNotFoundError("could not find repo root")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from train.stage1_oracle.features.control_v3 import (  # noqa: E402
    CONFIDENCE_FEATURE_NAMES,
    DEBUG_ARRAY_NAMES,
    FEATURE_NAMES,
    MODEL_FEATURE_NAMES,
    VALUE_FEATURE_NAMES,
    FeatureConfigV3,
    extract_control_features,
)
from train.stage1_oracle.features.control_v3_audit import (  # noqa: E402
    FEATURE_CONTRACT,
    feature_audit_summary,
    feature_contract_report,
    high_value_confidence_audit,
    evaluate_control_v3_audit,
    section_summaries_for_frame,
    saturation_report,
    stratified_review_queue,
)


In [2]:
DATASET_ROOT = REPO_ROOT / "mania-dataset"
INDEX_PATH = REPO_ROOT / "train/artifacts/indexes/beatmap_index_4k_no_timing_anomalies_2to6.parquet"
AUDIT_DIR = REPO_ROOT / "train/artifacts/features/control_v3_audit"
TIMESERIES_PATH = REPO_ROOT / "train/artifacts/features/control_v3_timeseries_4k_no_timing_anomalies_2to6.parquet"
METADATA_PATH = REPO_ROOT / "train/artifacts/features/control_v3_artifact_metadata_4k_no_timing_anomalies_2to6.json"
SECTION_AUDIT_PATH = AUDIT_DIR / "control_v3_section_audit_8s_stride4.parquet"

CFG = FeatureConfigV3(grid_step=0.10)
VALUE_FEATURES = list(VALUE_FEATURE_NAMES)
CONFIDENCE_FEATURES = list(CONFIDENCE_FEATURE_NAMES)
MODEL_CHANNELS = list(MODEL_FEATURE_NAMES)
DEBUG_ARRAY_COLUMNS = list(DEBUG_ARRAY_NAMES)

if FEATURE_NAMES != MODEL_FEATURE_NAMES:
    raise RuntimeError("control_v3 FEATURE_NAMES and MODEL_FEATURE_NAMES disagree")
if "ln_change_rate_gated" not in MODEL_CHANNELS:
    raise RuntimeError("v3 model channels must include ln_change_rate_gated")
if "ln_change_rate_raw" not in DEBUG_ARRAY_COLUMNS:
    raise RuntimeError("v3 diagnostics must include ln_change_rate_raw")

print("repo root:", REPO_ROOT)
print("timeseries parquet:", TIMESERIES_PATH)
print("metadata json:", METADATA_PATH)
print("v3 model channels:", len(MODEL_CHANNELS), MODEL_CHANNELS)
print("v3 contracts:", FEATURE_CONTRACT)


repo root: /Users/l/projects/Mapperatorinator
timeseries parquet: /Users/l/projects/Mapperatorinator/train/artifacts/features/control_v3_timeseries_4k_no_timing_anomalies_2to6.parquet
metadata json: /Users/l/projects/Mapperatorinator/train/artifacts/features/control_v3_artifact_metadata_4k_no_timing_anomalies_2to6.json
v3 model channels: 20 ['density_level', 'density_burst', 'hold_occupancy', 'ln_change_rate_gated', 'chord_ratio', 'jack_excess', 'jack_streak_exposure', 'hand_balance_signed', 'hand_imbalance_abs', 'repeat_exact', 'repeat_shift', 'repeat_motion', 'density_confidence', 'ln_change_confidence', 'chord_confidence', 'jack_confidence', 'jack_streak_confidence', 'hand_confidence', 'repeat_confidence', 'control_confidence']
v3 contracts: {'ln_change_rate_raw': 'raw_value_with_side_confidence', 'ln_change_rate_gated': 'confidence_gated_value', 'density_level': 'stable_value', 'density_burst': 'stable_value', 'hold_occupancy': 'stable_value', 'chord_ratio': 'confidence_gated_or_hi

In [3]:
def build_section_audit_from_timeseries_parquet(
    *,
    timeseries_path: Path = TIMESERIES_PATH,
    section_s: float = 8.0,
    stride_s: float = 4.0,
    output_path: Path | None = SECTION_AUDIT_PATH,
) -> pd.DataFrame:
    time_df = pd.read_parquet(timeseries_path)
    records = []
    for filtered_index, group in time_df.sort_values(["filtered_index", "time_s"]).groupby("filtered_index", sort=True):
        row = group.iloc[0]
        metadata_columns = ["filtered_index", "source_index", "beatmap_id", "beatmap_set_id", "difficulty"]
        frame = group.drop(columns=[column for column in metadata_columns if column in group])
        sections = section_summaries_for_frame(row, frame, section_s=section_s, stride_s=stride_s)
        if not sections.empty:
            sections["filtered_index"] = int(filtered_index)
            for column in ["beatmap_id", "beatmap_set_id", "difficulty"]:
                if column in group:
                    sections[column] = group[column].iloc[0]
            records.append(sections)
    section_df = pd.concat(records, ignore_index=True) if records else pd.DataFrame()
    if output_path is not None:
        output_path.parent.mkdir(parents=True, exist_ok=True)
        section_df.to_parquet(output_path, index=False)
    return section_df


In [4]:
artifact_metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8")) if METADATA_PATH.exists() else {}
if artifact_metadata:
    print("artifact schema_version:", artifact_metadata.get("schema_version"))
    print("feature_contract_version:", artifact_metadata.get("feature_contract_version"))
    print("artifact map_count:", artifact_metadata.get("map_count"))
    print("artifact timeseries_rows:", artifact_metadata.get("timeseries_rows"))
    print("artifact error_count:", artifact_metadata.get("error_count"))
    print("artifact model channels:", len(artifact_metadata.get("model_feature_names", [])), artifact_metadata.get("model_feature_names", []))
else:
    print("missing metadata json:", METADATA_PATH)

if TIMESERIES_PATH.exists():
    section_df = build_section_audit_from_timeseries_parquet()
    print("section rows:", len(section_df))
    if "filtered_index" in section_df:
        print("section unique maps:", section_df["filtered_index"].nunique())
    display(feature_contract_report())
    display(saturation_report(section_df))
    display(feature_audit_summary(section_df))
    display(high_value_confidence_audit(section_df).head(25))
    display(evaluate_control_v3_audit(section_df))
    display(stratified_review_queue(section_df, n_per_bucket=25))
else:
    print("missing timeseries parquet:", TIMESERIES_PATH)


artifact schema_version: 3
feature_contract_version: 3
artifact map_count: 10977
artifact timeseries_rows: 16858417
artifact error_count: 0
artifact model channels: 20 ['density_level', 'density_burst', 'hold_occupancy', 'ln_change_rate_gated', 'chord_ratio', 'jack_excess', 'jack_streak_exposure', 'hand_balance_signed', 'hand_imbalance_abs', 'repeat_exact', 'repeat_shift', 'repeat_motion', 'density_confidence', 'ln_change_confidence', 'chord_confidence', 'jack_confidence', 'jack_streak_confidence', 'hand_confidence', 'repeat_confidence', 'control_confidence']


section rows: 407491
section unique maps: 10977


,feature,contract,scope,audit_class
0,ln_change_rate_gated,confidence_gated_value,model,
1,density_level,stable_value,model,
2,density_burst,stable_value,model,
3,hold_occupancy,stable_value,model,
4,chord_ratio,confidence_gated_or_high_support,model,
5,jack_excess,sparse_tail_with_confidence,model,
6,jack_streak_exposure,support_sensitive,model,
7,hand_balance_signed,confidence_gated_value,model,
8,hand_imbalance_abs,confidence_gated_value,model,
9,repeat_exact,support_sensitive,model,


,feature,stat,count,p95,p99,max,near_zero_rate,near_high_rate,tail_heaviness,max_to_p99,low_threshold,high_threshold
0,jack_excess,p95,407491,0.058731,0.278366,0.903495,NaN,NaN,4.739704,3.245712,NaN,NaN
1,hand_balance_signed,p95,407491,0.130050,0.216825,0.815342,NaN,NaN,1.667247,3.760378,NaN,NaN
2,hold_occupancy,p95,407491,0.450625,0.612771,1.000000,NaN,NaN,1.359823,1.631931,NaN,NaN
3,ln_change_rate_gated,p95,407491,3.079284,3.465537,4.128909,NaN,NaN,1.125436,1.191420,NaN,NaN
4,jack_streak_exposure,p95,407491,0.901683,0.987575,0.999980,NaN,NaN,1.095258,1.012561,NaN,NaN
5,density_level,p95,407491,3.178216,3.317422,3.513811,NaN,NaN,1.043800,1.059199,NaN,NaN
6,density_burst,p95,407491,0.999954,0.999999,1.000000,NaN,NaN,1.000046,1.000001,NaN,NaN
7,repeat_shift,p95,407491,0.338482,0.480584,0.999870,0.002790,0.000044,NaN,NaN,0.01,0.95
8,repeat_exact,p95,407491,0.194230,0.308084,0.999870,0.002834,0.000037,NaN,NaN,0.01,0.95
9,repeat_motion,p95,407491,0.242455,0.368261,0.999870,0.002805,0.000037,NaN,NaN,0.01,0.95


,feature,contract,high_value_section_count,high_value_section_rate,pointwise_high_value_low_conf_rate,window_only_low_conf_rate,low_support_high_value_rate,confidence_at_peak_p10,confidence_at_peak_median,section_confidence_p20_median,high_value_unique_map_rate,high_value_difficulty_mean,pointwise_unique_map_rate,window_only_unique_map_rate,low_support_unique_map_rate
0,density_level,stable_value,242075,0.594062,0.000000,0.002204,0.000000,0.999937,1.000000,0.999998,0.768971,4.323320,0.000000,0.063314,0.000000
1,density_burst,stable_value,397663,0.975882,0.001453,0.006668,0.000000,0.999646,0.999999,0.999998,1.000000,3.778967,0.021955,0.146124,0.000000
2,ln_change_rate_gated,confidence_gated_value,255778,0.627690,0.000000,0.393506,0.035024,0.000000,0.805172,0.000000,0.893231,3.690217,0.000000,0.876742,0.401294
3,chord_ratio,confidence_gated_or_high_support,9558,0.023456,0.000000,0.000537,0.000000,0.862278,0.979397,0.974604,0.137834,4.815706,0.000000,0.014029,0.000000
4,jack_excess,sparse_tail_with_confidence,1004,0.002464,0.000000,0.000898,0.000000,0.000000,0.810832,0.727968,0.027057,4.393008,0.000000,0.013665,0.000000
5,jack_streak_exposure,support_sensitive,66237,0.162548,0.000000,0.047248,0.000000,0.000000,0.526603,0.000000,0.338708,5.112609,0.000000,0.307097,0.000000
6,hand_imbalance_abs,confidence_gated_value,480,0.001178,0.000000,0.000039,0.000000,0.603239,0.762777,0.751836,0.011843,4.145208,0.000000,0.000638,0.000000
7,repeat_exact,support_sensitive,904,0.002218,0.000000,0.000135,0.000000,0.964407,0.998589,0.997523,0.026965,4.178540,0.000000,0.003097,0.000000
8,repeat_shift,support_sensitive,3264,0.008010,0.000000,0.000452,0.000000,0.965376,0.998737,0.997523,0.093377,3.885959,0.000000,0.011752,0.000000
9,repeat_motion,support_sensitive,1387,0.003404,0.000000,0.000191,0.000000,0.964547,0.998582,0.997523,0.042361,4.070000,0.000000,0.004646,0.000000


,filtered_index,beatmap_id,difficulty,section_start_s,section_end_s,valid_fraction,control_confidence_mean,feature,contract,audit_class,...,value_threshold,confidence_at_peak,confidence_top_value_mean,confidence_top_value_min,confidence_section_p20,confidence_section_min,confidence_threshold,n_eff_at_peak,n_eff_top_value_mean,n_eff_threshold
0,1953.0,2968556.0,4.18,64.0,72.0,1.0000,0.694285,ln_change_rate_gated,confidence_gated_value,low_support_high_value,...,0.5,0.629026,0.656441,0.624178,0.000000,0.000000,0.2,2.983247,3.160305,3.0
1,1953.0,2968556.0,4.18,68.0,76.0,1.0000,0.726107,ln_change_rate_gated,confidence_gated_value,low_support_high_value,...,0.5,0.629026,0.656441,0.624178,0.389986,0.376131,0.2,2.983247,3.160305,3.0
2,9228.0,1122126.0,3.91,236.0,244.0,1.0000,0.459156,ln_change_rate_gated,confidence_gated_value,low_support_high_value,...,0.5,0.631311,0.630207,0.629381,0.000000,0.000000,0.2,2.995605,2.989626,3.0
3,7579.0,4913933.0,4.82,68.0,76.0,1.0000,0.817869,ln_change_rate_gated,confidence_gated_value,low_support_high_value,...,0.5,0.630840,0.630112,0.628398,0.000000,0.000000,0.2,2.993049,2.989117,3.0
4,7579.0,4913933.0,4.82,72.0,80.0,1.0000,0.799306,ln_change_rate_gated,confidence_gated_value,low_support_high_value,...,0.5,0.630840,0.630112,0.628398,0.000000,0.000000,0.2,2.993049,2.989117,3.0
5,4.0,2156259.0,3.39,4.0,12.0,1.0000,0.746778,ln_change_rate_gated,confidence_gated_value,low_support_high_value,...,0.5,0.627876,0.628436,0.627876,0.000000,0.000000,0.2,2.977054,2.980071,3.0
6,9335.0,1159134.0,5.21,16.0,24.0,1.0000,0.678031,ln_change_rate_gated,confidence_gated_value,low_support_high_value,...,0.5,0.628395,0.626385,0.621572,0.000000,0.000000,0.2,2.979848,2.969167,3.0
7,9335.0,1159134.0,5.21,20.0,28.0,1.0000,0.804529,ln_change_rate_gated,confidence_gated_value,low_support_high_value,...,0.5,0.628395,0.626385,0.621572,0.000000,0.000000,0.2,2.979848,2.969167,3.0
8,4.0,2156259.0,3.39,0.0,8.0,0.9750,0.738855,ln_change_rate_gated,confidence_gated_value,low_support_high_value,...,0.5,0.627876,0.627836,0.626402,0.000000,0.000000,0.2,2.977054,2.976847,3.0
9,9796.0,1368120.0,2.64,16.0,24.0,1.0000,0.655807,ln_change_rate_gated,confidence_gated_value,low_support_high_value,...,0.5,0.628814,0.628448,0.627553,0.000000,0.000000,0.2,2.982102,2.980133,3.0


,gate_name,metric,value,threshold,pass,severity,reason,feature,high_value_section_count,failure_count,confidence,feature_section_count
0,finite_model_stats,finite_rate,1.000000,>= 0.999,True,hard_gate,model section statistics should be finite,NaN,NaN,NaN,NaN,NaN
1,valid_fraction,mean,0.993986,>= 0.80,True,hard_gate,section windows should mostly cover valid map ...,NaN,NaN,NaN,NaN,NaN
2,required_feature_aligned_diagnostics_present,missing_column_count,0.000000,== 0,True,hard_gate,all high-value audited features must carry pea...,NaN,NaN,NaN,NaN,NaN
3,pointwise_high_value_low_confidence,feature_section_rate,0.000145,<= 0.02,True,hard_gate,hard failures require high value to align with...,NaN,NaN,NaN,NaN,NaN
4,low_support_high_value,feature_section_rate,0.003502,<= 0.02,True,hard_gate,hard failures require high value to align with...,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
98,repeat_exact_near_zero,near_zero_rate,0.002834,<= 0.98,True,hard_gate,repeat channels should not be structurally zero,NaN,NaN,NaN,NaN,NaN
99,repeat_shift_near_high,near_high_rate,0.000044,<= 0.05,True,hard_gate,bounded channels should not saturate high acro...,NaN,NaN,NaN,NaN,NaN
100,repeat_shift_near_zero,near_zero_rate,0.002790,<= 0.98,True,hard_gate,repeat channels should not be structurally zero,NaN,NaN,NaN,NaN,NaN
101,repeat_motion_near_high,near_high_rate,0.000037,<= 0.05,True,hard_gate,bounded channels should not saturate high acro...,NaN,NaN,NaN,NaN,NaN


,review_bucket,audit_class,feature,contract,value,confidence_at_peak,confidence_top_value_mean,confidence_section_p20,confidence_section_min,n_eff_at_peak,n_eff_top_value_mean,filtered_index,beatmap_id,difficulty,section_start_s,section_end_s
0,pointwise_high_value_low_confidence,pointwise_high_value_low_confidence,density_burst,stable_value,0.999818,0.999998,0.749998,0.143181,0.000000,8.580961,6.713335,2705.0,3316779.0,4.19,0.0,8.0
1,pointwise_high_value_low_confidence,pointwise_high_value_low_confidence,density_burst,stable_value,0.999325,1.000000,0.597375,0.252270,0.170706,11.769192,6.356732,10399.0,2046467.0,4.13,108.0,116.0
2,pointwise_high_value_low_confidence,pointwise_high_value_low_confidence,density_burst,stable_value,0.999324,0.170706,0.183719,0.192409,0.170706,1.000000,1.000000,10399.0,2046467.0,4.13,112.0,120.0
3,pointwise_high_value_low_confidence,pointwise_high_value_low_confidence,density_burst,stable_value,0.999261,0.000000,0.190689,0.000000,0.000000,1.000000,1.013504,7884.0,563161.0,3.98,116.0,124.0
4,pointwise_high_value_low_confidence,pointwise_high_value_low_confidence,density_burst,stable_value,0.998985,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,3497.0,3621892.0,5.10,0.0,8.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
120,ln_change_case,ln_change_case,ln_change_rate_gated,NaN,3.893325,NaN,NaN,NaN,NaN,NaN,NaN,8465.0,879298.0,5.13,216.0,224.0
121,ln_change_case,ln_change_case,ln_change_rate_gated,NaN,3.891959,NaN,NaN,NaN,NaN,NaN,NaN,8465.0,879298.0,5.13,212.0,220.0
122,ln_change_case,ln_change_case,ln_change_rate_gated,NaN,3.890977,NaN,NaN,NaN,NaN,NaN,NaN,8465.0,879298.0,5.13,220.0,228.0
123,ln_change_case,ln_change_case,ln_change_rate_gated,NaN,3.885015,NaN,NaN,NaN,NaN,NaN,NaN,8924.0,1076362.0,5.78,248.0,256.0
